# Architecture research progress after the multisplit frontier

This aggregate-only notebook summarizes two completed AWS neural studies and one measured GPU data-loading benchmark. It contains no raw tracking data, fitted weights, checkpoint locations, or unreleased feature transforms.

In [1]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

reference_rmse = 0.4537257476309679
studies = pd.DataFrame(
    [
        {
            "family": "Expanded observed-motion representation",
            "standalone_rmse": 0.46063278603812063,
            "blend_rmse": 0.4525854218277901,
            "gain": 0.0011403258031777996,
            "ci_low": -0.00040696886555451756,
            "ci_high": 0.0024975173169987526,
        },
        {
            "family": "Future-conditioned delta decoder",
            "standalone_rmse": 0.4599762248441056,
            "blend_rmse": 0.4529808663099405,
            "gain": 0.0007448813210274019,
            "ci_low": -0.0006815486872915472,
            "ci_high": 0.0020490253736768337,
        },
    ]
)
studies["decision"] = "NO_PROMOTION"

In [2]:
display(studies)
assert (studies["gain"] < 0.0015).all()
assert (studies["ci_low"] < 0).all()

                                      family  standalone_rmse  blend_rmse      gain    ci_low   ci_high      decision
0  Expanded observed-motion representation         0.460633    0.452585  0.001140 -0.000407  0.002498  NO_PROMOTION
1         Future-conditioned delta decoder         0.459976    0.452981  0.000745 -0.000682  0.002049  NO_PROMOTION

In [3]:
fig = go.Figure()
fig.add_bar(
    x=studies["family"],
    y=studies["blend_rmse"],
    name="Fixed 80/20 blend",
)
fig.add_scatter(
    x=studies["family"],
    y=[reference_rmse] * len(studies),
    mode="markers",
    name="Fixed reference",
)
fig.update_layout(
    title="Controlled Fold-0 ensemble comparisons",
    yaxis_title="Coordinate RMSE",
    width=900,
    height=480,
    template=None,
)
fig.show()

In [4]:
loader = pd.DataFrame(
    [
        {"workers": 0, "seconds": 0.11017817699939769},
        {"workers": 2, "seconds": 0.04855473699990398},
        {"workers": 4, "seconds": 0.04861624949990073},
        {"workers": 12, "seconds": 0.05089610849972814},
    ]
)
loader["relative_speedup"] = loader.loc[0, "seconds"] / loader["seconds"]
display(loader)

   workers   seconds  relative_speedup
0        0  0.110178          1.000000
1        2  0.048555          2.269136
2        4  0.048616          2.266265
3       12  0.050896          2.164775

In [5]:
fig = go.Figure(
    go.Bar(
        x=loader["workers"].astype(str),
        y=loader["seconds"],
        text=loader["seconds"].map(lambda value: f"{value:.4f}s"),
        textposition="outside",
    )
)
fig.update_layout(
    title="Measured full training-step time on the current L4",
    xaxis_title="Loader workers",
    yaxis_title="Median seconds per step",
    width=900,
    height=480,
    template=None,
)
fig.show()
print("ARCHITECTURE_PROGRESS_SNAPSHOT_COMPLETE")

ARCHITECTURE_PROGRESS_SNAPSHOT_COMPLETE


## Publication boundary

The notebook exposes aggregate metrics, locked decisions, and measured runtime behavior. Exact private runners, fitted states, object locations, and unreleased feature/interaction transforms remain in the AWS research workspace.